# Layer 5: Classification and Anomaly Detection

This notebook runs the full L5 pipeline and summarizes outputs.
It uses fused signals from L1 + L2 + L4 through the scripts in L5_Classification.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = Path.cwd()
L5_DIR = ROOT / 'L5_Classification'
OUT_DIR = L5_DIR / 'outputs'
PLOT_DIR = L5_DIR / 'plots'

print('Root:', ROOT)
print('L5 dir exists:', L5_DIR.exists())

In [ ]:
# Step 1: Build fused L5 feature table
subprocess.run([sys.executable, str(L5_DIR / '01_build_feature_table.py')], check=True)

meta_path = OUT_DIR / 'l5_build_metadata.json'
meta = json.loads(meta_path.read_text(encoding='utf-8'))
print(json.dumps(meta, indent=2))

df_features = pd.read_csv(OUT_DIR / 'l5_feature_table.csv')
df_features.head()

In [ ]:
# Step 2: Train supervised and anomaly models
subprocess.run([sys.executable, str(L5_DIR / '02_train_models.py')], check=True)
subprocess.run([sys.executable, str(L5_DIR / '03_anomaly_detection.py')], check=True)
print('Training complete.')

In [ ]:
# Step 3: Metrics summary
supervised_metrics = pd.read_csv(OUT_DIR / 'supervised_model_metrics.csv')
anomaly_metrics = pd.read_csv(OUT_DIR / 'anomaly_model_metrics.csv')

print('Supervised models:')
display(supervised_metrics)

print('Anomaly models:')
display(anomaly_metrics)

In [ ]:
# Step 4: Best supervised model
best_model_name = (OUT_DIR / 'supervised_best_model_name.txt').read_text(encoding='utf-8').strip()
print('Best supervised model by AUC:', best_model_name)

In [ ]:
# Step 5: Plot artifacts
plot_files = [
    'l5_supervised_roc.png',
    'l5_supervised_pr.png',
    'l5_random_forest_feature_importance.png',
    'l5_anomaly_roc.png',
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, fname in zip(axes, plot_files):
    img = mpimg.imread(PLOT_DIR / fname)
    ax.imshow(img)
    ax.set_title(fname)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Notes
- Base feature source is auto-selected by L5 builder and stored in `l5_build_metadata.json`.
- Current script priority is: `reviewer_features.csv` -> `L1_ETL_OLAP/output_csv/reviewer_profiles.csv` -> `reviewer_profiles.csv`.